In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torch.nn import Conv2d, MaxPool2d

# 1. 准备数据管道 (CIFAR-10 测试集 10000 张图，每次发 1 张)
dataset = torchvision.datasets.CIFAR10("../data", train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=1)

# 2. 搭建网络模型 (层级结构已掌握，此处不作冗余注释)
class Model(nn.Module):
    def __init__(self):
        super().__init__() # 现代化初始化
        self.model1 = nn.Sequential(
            Conv2d(3, 32, 5, padding=2), MaxPool2d(2),
            Conv2d(32, 32, 5, padding=2), MaxPool2d(2),
            Conv2d(32, 64, 5, padding=2), MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        return self.model1(x)

# 3. 实例化模型、高精度损失函数与优化器
test_model = Model()
loss_fn = nn.CrossEntropyLoss()
# 声明随机梯度下降（SGD）优化器，绑定模型参数，学习率设为 0.01
optimizer = torch.optim.SGD(test_model.parameters(), lr=0.01)

# 4. 训练循环 (只训练 1 轮)
for data in dataloader:
    imgs, targets = data
    outputs = test_model(imgs)             # 前向传播：计算预测 Logits
    loss_result = loss_fn(outputs, targets) # 计算损失

    # 优化器标准三步曲
    optimizer.zero_grad()       # 1) 历史梯度归零
    loss_result.backward()      # 2) 反向传播计算当前梯度
    optimizer.step()            # 3) 根据梯度更新网络参数 w 和 b

    # 核心：在 for 循环内部，每处理 1 张图，就打印一次当前单步的 Loss
    print(loss_result)

100%|██████████| 170M/170M [00:15<00:00, 11.0MB/s]


流式输出内容被截断，只能显示最后 5000 行内容。
tensor(1.8248, grad_fn=<NllLossBackward0>)
tensor(0.7039, grad_fn=<NllLossBackward0>)
tensor(1.9369, grad_fn=<NllLossBackward0>)
tensor(1.1907, grad_fn=<NllLossBackward0>)
tensor(1.2818, grad_fn=<NllLossBackward0>)
tensor(3.8445, grad_fn=<NllLossBackward0>)
tensor(2.7928, grad_fn=<NllLossBackward0>)
tensor(1.6997, grad_fn=<NllLossBackward0>)
tensor(1.4159, grad_fn=<NllLossBackward0>)
tensor(1.2789, grad_fn=<NllLossBackward0>)
tensor(0.4591, grad_fn=<NllLossBackward0>)
tensor(2.3385, grad_fn=<NllLossBackward0>)
tensor(2.2343, grad_fn=<NllLossBackward0>)
tensor(4.2618, grad_fn=<NllLossBackward0>)
tensor(0.2926, grad_fn=<NllLossBackward0>)
tensor(1.5522, grad_fn=<NllLossBackward0>)
tensor(3.1151, grad_fn=<NllLossBackward0>)
tensor(2.5182, grad_fn=<NllLossBackward0>)
tensor(2.3368, grad_fn=<NllLossBackward0>)
tensor(0.7352, grad_fn=<NllLossBackward0>)
tensor(1.4446, grad_fn=<NllLossBackward0>)
tensor(0.6688, grad_fn=<NllLossBackward0>)
tensor(1.6685, grad_fn=<Nll

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torch.nn import Conv2d, MaxPool2d

# 1. 准备数据管道 (CIFAR-10 测试集 10000 张图，每次发 1 张)
dataset = torchvision.datasets.CIFAR10("../data", train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=1)

# 2. 搭建网络模型 (层级结构已掌握，此处不作冗余注释)
class Model(nn.Module):
    def __init__(self):
        super().__init__() # 现代化初始化
        self.model1 = nn.Sequential(
            Conv2d(3, 32, 5, padding=2), MaxPool2d(2),
            Conv2d(32, 32, 5, padding=2), MaxPool2d(2),
            Conv2d(32, 64, 5, padding=2), MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        return self.model1(x)

# 3. 实例化模型、高精度损失函数与优化器
test_model = Model()
loss_fn = nn.CrossEntropyLoss()
# 声明随机梯度下降（SGD）优化器，绑定模型参数，学习率设为 0.01
optimizer = torch.optim.SGD(test_model.parameters(), lr=0.01)

# 4. 训练循环 (引入 Epoch 外循环与 Loss 累加)
# epochs=20，代表让模型将 10000 张图片完整地训练（学习）20 遍
for epoch in range(20):
    running_loss = 0.0  # 每一轮（Epoch）开始前，初始化本轮的总损失为 0

    for data in dataloader:
        imgs, targets = data
        outputs = test_model(imgs)
        loss_result = loss_fn(outputs, targets)

        optimizer.zero_grad()
        loss_result.backward()
        optimizer.step()

        # 核心：在内部循环中，不打印单步 Loss，而是将其不断累加到本轮总损失中
        running_loss = running_loss + loss_result

    # 在外层循环（即每张卡片完整训练完一遍后），只打印一次本轮的总损失值
    print(running_loss)

tensor(18737.8750, grad_fn=<AddBackward0>)
tensor(16148.6504, grad_fn=<AddBackward0>)
tensor(15469.3613, grad_fn=<AddBackward0>)
tensor(15900.6396, grad_fn=<AddBackward0>)
tensor(17800.1094, grad_fn=<AddBackward0>)
tensor(19989.6074, grad_fn=<AddBackward0>)
tensor(22409.5488, grad_fn=<AddBackward0>)
tensor(23279.3516, grad_fn=<AddBackward0>)
tensor(24484.1445, grad_fn=<AddBackward0>)


KeyboardInterrupt: 